# Tulane uptown campus — real-data 3D model
Builds a to-scale glTF model from public survey data: OpenStreetMap footprints, USGS 3DEP LiDAR (the laser scan of New Orleans) and USDA NAIP aerial imagery.

**Runtime → Run all.** Steps 1–3 take a couple of minutes; step 4 downloads a few hundred MB of LiDAR and builds the model (5–15 min). Step 5 saves everything to your Google Drive.

## 1. Write the pipeline files

In [ ]:
import os, textwrap
os.makedirs('/content/tulane-scan/data/lidar', exist_ok=True); os.chdir('/content/tulane-scan')
FILES = {
"requirements.txt": "numpy>=1.24\nscipy>=1.10\nshapely>=2.0\npyproj>=3.4\nlaspy[lazrs]>=2.4\ntrimesh>=4.0\nmapbox_earcut>=1.0\npillow>=9.0\nrequests>=2.28\nosm2geojson>=0.2\nmatplotlib>=3.6\n",
"fetch_osm.py": "#!/usr/bin/env python3\n\"\"\"Download OpenStreetMap footprints, streets, streetcar line, trees and parks for the bbox -> data/osm.geojson\"\"\"\nimport argparse, json, os, sys, time, requests, osm2geojson\nap = argparse.ArgumentParser(); ap.add_argument('--bbox', default='29.9330,-90.1290,29.9500,-90.1120', help='south,west,north,east'); ap.add_argument('--out', default='data/osm.geojson'); a = ap.parse_args()\nb = a.bbox\nq = f'''[out:json][timeout:180];\n( way[\"building\"]({b}); relation[\"building\"]({b}); way[\"highway\"]({b}); way[\"railway\"=\"tram\"]({b}); node[\"natural\"=\"tree\"]({b});\n  way[\"leisure\"]({b}); way[\"landuse\"=\"grass\"]({b}); way[\"amenity\"=\"parking\"]({b}); way[\"natural\"=\"water\"]({b}); );\nout body; >; out skel qt;'''\nfor url in ['https://overpass-api.de/api/interpreter', 'https://overpass.kumi.systems/api/interpreter']:\n    try:\n        r = requests.post(url, data={'data': q}, timeout=300); r.raise_for_status(); data = r.json(); break\n    except Exception as e:\n        print('overpass mirror failed:', url, e); time.sleep(3)\nelse:\n    sys.exit('Overpass unavailable \u2014 retry later, or export GeoJSON from overpass-turbo.eu and save it as data/osm.geojson')\ngeo = osm2geojson.json2geojson(data)\nos.makedirs(os.path.dirname(a.out) or '.', exist_ok=True); json.dump(geo, open(a.out, 'w'))\nn = sum(1 for f in geo['features'] if (f['properties'].get('tags') or {}).get('building'))\nprint(f\"saved {a.out}: {len(geo['features'])} features, {n} buildings\")\n",
"fetch_naip.py": "#!/usr/bin/env python3\n\"\"\"Download a public-domain USDA NAIP aerial orthophoto of the bbox from USGS -> data/naip.png (+ .json with the bbox)\"\"\"\nimport argparse, json, os, sys, requests\nap = argparse.ArgumentParser(); ap.add_argument('--bbox', default='-90.1290,29.9330,-90.1120,29.9500', help='minLon,minLat,maxLon,maxLat'); ap.add_argument('--size', type=int, default=4096); ap.add_argument('--out', default='data/naip.png'); a = ap.parse_args()\nbbox = [float(v) for v in a.bbox.split(',')]\nservices = ['https://imagery.nationalmap.gov/arcgis/rest/services/USGSNAIPPlus/ImageServer/exportImage',\n            'https://basemap.nationalmap.gov/arcgis/rest/services/USGSImageryOnly/MapServer/export']\nfor url in services:\n    try:\n        r = requests.get(url, params={'bbox': a.bbox, 'bboxSR': 4326, 'imageSR': 4326, 'size': f'{a.size},{a.size}', 'format': 'png', 'f': 'image'}, timeout=300)\n        if r.ok and r.headers.get('content-type', '').startswith('image'):\n            os.makedirs(os.path.dirname(a.out) or '.', exist_ok=True); open(a.out, 'wb').write(r.content)\n            json.dump({'bbox': bbox, 'size': [a.size, a.size], 'source': url, 'license': 'USDA NAIP, public domain'}, open(os.path.splitext(a.out)[0] + '.json', 'w'))\n            print('saved', a.out, len(r.content) // 1024, 'KB from', url); sys.exit(0)\n        print('no image from', url, r.status_code)\n    except Exception as e: print('failed', url, e)\nsys.exit('could not fetch imagery; the model still builds without it (untextured terrain)')\n",
"fetch_lidar.py": "#!/usr/bin/env python3\n\"\"\"Find and download USGS 3DEP LiDAR point-cloud tiles (public domain) covering the bbox -> data/lidar/*.laz\nUses The National Map access API. Tiles are ~100-400 MB each; a campus bbox usually needs 2-6 tiles.\"\"\"\nimport argparse, os, sys, requests\nap = argparse.ArgumentParser(); ap.add_argument('--bbox', default='-90.1290,29.9330,-90.1120,29.9500', help='minLon,minLat,maxLon,maxLat')\nap.add_argument('--out', default='data/lidar'); ap.add_argument('--project', default=None, help='only tiles whose title contains this text'); ap.add_argument('--list', action='store_true'); ap.add_argument('--yes', action='store_true'); a = ap.parse_args()\nAPI = 'https://tnmaccess.nationalmap.gov/api/v1/products'\nr = requests.get(API, params={'datasets': 'Lidar Point Cloud (LPC)', 'bbox': a.bbox, 'prodFormats': 'LAZ', 'outputFormat': 'JSON', 'max': 500}, timeout=180); r.raise_for_status()\nitems = r.json().get('items', [])\nif a.project: items = [i for i in items if a.project.lower() in (i.get('title') or '').lower()]\nif not items: sys.exit('no LiDAR tiles returned for this bbox \u2014 try https://apps.nationalmap.gov/downloader/ (Elevation Source Data) or Louisiana\\'s atlas.ga.lsu.edu')\nproj = {}\nfor i in items: proj.setdefault(i.get('title', '?').split(' ')[0], []).append(i)\nprint(f'{len(items)} tiles from {len(proj)} project(s):')\nfor k, v in proj.items(): print(f'  {k}: {len(v)} tiles, published {v[0].get(\"publicationDate\")}, {sum((i.get(\"sizeInBytes\") or 0) for i in v) / 1e6:.0f} MB, e.g. {v[0].get(\"title\")}')\nif a.list: sys.exit(0)\nnewest = max(proj.values(), key=lambda v: str(v[0].get('publicationDate')))\nif not a.project and len(proj) > 1: print('downloading the newest project only (use --project to pick another)'); items = newest\ntotal = sum((i.get('sizeInBytes') or 0) for i in items)\nif not a.yes and input(f'download {len(items)} tiles ({total / 1e6:.0f} MB)? [y/N] ').lower() != 'y': sys.exit(0)\nos.makedirs(a.out, exist_ok=True)\nfor i in items:\n    url = i['downloadURL']; fn = os.path.join(a.out, os.path.basename(url.split('?')[0]))\n    if os.path.exists(fn): print('have', fn); continue\n    print('downloading', fn)\n    with requests.get(url, stream=True, timeout=600) as s:\n        s.raise_for_status()\n        with open(fn + '.part', 'wb') as f:\n            for chunk in s.iter_content(1 << 20): f.write(chunk)\n    os.rename(fn + '.part', fn)\nprint('done')\n",
"build_model.py": "#!/usr/bin/env python3\n\"\"\"\nbuild_model.py \u2014 turn real survey data into a game-ready 3D model of the Tulane uptown campus.\n\nInputs (all free / public):\n  * data/osm.geojson        OSM building footprints, streets etc. (fetch_osm.py) \u2014 or an Overture buildings GeoJSON\n  * data/lidar/*.laz        USGS 3DEP LiDAR point clouds (fetch_lidar.py)  [optional but this is the \"scan\"]\n  * data/naip.png + .json   NAIP aerial orthophoto of the bbox (fetch_naip.py) [optional, used as texture/colors]\n\nOutputs (out/):\n  tulane_buildings.glb      one mesh per building; walls at LiDAR-measured height, roof surface from the LiDAR DSM\n  tulane_terrain.glb        ground surface from the LiDAR ground returns, textured with the orthophoto\n  tulane_trees.glb + .json  tree instances detected from the LiDAR canopy (position, height, crown radius)\n  tulane_scan_surface.glb   optional (--surface): the raw scanned surface (ground+buildings+trees) as one textured mesh\n  tulane_enriched.geojson   the footprints with measured `height`, `roof:height` etc. \u2014 loads straight into the voxel viewer\n  preview_dsm.png           hillshaded scan with footprints, for a sanity check\n\nCoordinate frame: meters, Y up, X east, Z south (glTF convention). Optional --align-st-charles rotates so\nSt. Charles Avenue runs along +X (the same frame the voxel viewer uses).\n\"\"\"\nimport argparse, glob, json, math, os, sys, time\nimport numpy as np\n\ndef log(*a):\n    print(time.strftime('%H:%M:%S'), *a, flush=True)\n\n# ----------------------------------------------------------------------------- args\nap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)\nap.add_argument('--osm', default='data/osm.geojson', help='GeoJSON with building footprints (OSM via osm2geojson, Overpass Turbo export, or Overture)')\nap.add_argument('--lidar', default='data/lidar', help='directory of .laz/.las files, or a single file')\nap.add_argument('--naip', default='data/naip.png', help='orthophoto PNG (with a sidecar .json holding its lon/lat bbox)')\nap.add_argument('--bbox', default=None, help='minLon,minLat,maxLon,maxLat (default: bbox of the footprint data)')\nap.add_argument('--out', default='out')\nap.add_argument('--res', type=float, default=1.0, help='roof/DSM grid resolution in m (0.5 for finer roofs)')\nap.add_argument('--terrain-res', type=float, default=2.0, help='terrain grid resolution in m')\nap.add_argument('--surface', action='store_true', help='also export the whole scanned surface as one mesh')\nap.add_argument('--surface-res', type=float, default=2.0)\nap.add_argument('--align-st-charles', action='store_true', help='rotate the frame so St. Charles Ave runs along +X')\nap.add_argument('--lidar-epsg', type=int, default=None, help='override the point cloud CRS if the files lack one')\nap.add_argument('--z-scale', type=float, default=None, help='multiply LiDAR z by this (0.3048 if elevations are in feet); auto by default')\nap.add_argument('--max-points', type=int, default=60_000_000, help='random-subsample the cloud above this many points')\nap.add_argument('--max-trees', type=int, default=6000)\nap.add_argument('--obj', action='store_true', help='also write .obj copies of the glb files')\nargs = ap.parse_args()\nos.makedirs(args.out, exist_ok=True)\n\nfrom pyproj import Transformer, CRS\nimport shapely\nfrom shapely.geometry import shape, Polygon, MultiPolygon\nfrom scipy import ndimage\nimport trimesh\n\n# ----------------------------------------------------------------------------- footprints\ndef load_geojson(path):\n    with open(path, 'r', encoding='utf-8') as f:\n        gj = json.load(f)\n    feats = gj['features'] if gj.get('type') == 'FeatureCollection' else [gj]\n    return [f for f in feats if f.get('geometry')]\n\ndef props_of(f):\n    \"\"\"Normalize OSM (Overpass Turbo / osm2geojson) and Overture properties into one dict.\"\"\"\n    p = f.get('properties') or {}\n    t = p.get('tags') if isinstance(p.get('tags'), dict) else p\n    name = t.get('name') or (p.get('names') or {}).get('primary') if isinstance(p.get('names'), dict) else t.get('name')\n    building = t.get('building') or p.get('subtype') or p.get('class') or ('yes' if 'height' in p or 'num_floors' in p else None)\n    height = t.get('height', p.get('height'))\n    levels = t.get('building:levels', p.get('num_floors'))\n    fid = p.get('id') or p.get('@id') or f.get('id')\n    return dict(name=name, building=building, height=_num(height), levels=_num(levels),\n                roof_shape=t.get('roof:shape') or p.get('roof_shape'), tags=t, id=fid)\n\ndef _num(v):\n    if v is None: return None\n    try:\n        return float(str(v).replace('m', '').replace(',', '.').strip().split(' ')[0])\n    except Exception:\n        return None\n\nfeats = load_geojson(args.osm)\nlog(f'{len(feats)} features in {args.osm}')\n\ndef walk(coords, fn):\n    if isinstance(coords[0], (int, float)): fn(coords)\n    else:\n        for c in coords: walk(c, fn)\n\nif args.bbox:\n    minLon, minLat, maxLon, maxLat = [float(v) for v in args.bbox.split(',')]\nelse:\n    lons, lats = [], []\n    for f in feats: walk(f['geometry']['coordinates'], lambda c: (lons.append(c[0]), lats.append(c[1])))\n    minLon, maxLon, minLat, maxLat = min(lons), max(lons), min(lats), max(lats)\nlog(f'bbox lon {minLon:.5f}..{maxLon:.5f} lat {minLat:.5f}..{maxLat:.5f}')\n\n# ----------------------------------------------------------------------------- local frame\nclass Frame:\n    \"\"\"Local metric frame: origin at the bbox center, +y north (or rotated), computed through UTM 15N.\"\"\"\n    def __init__(self, lon0, lat0, theta=math.pi / 2):\n        self.utm = CRS.from_epsg(32615)\n        self.fwd = Transformer.from_crs(CRS.from_epsg(4326), self.utm, always_xy=True)\n        self.inv = Transformer.from_crs(self.utm, CRS.from_epsg(4326), always_xy=True)\n        self.E0, self.N0 = self.fwd.transform(lon0, lat0)\n        self.set_theta(theta)\n    def set_theta(self, theta):\n        self.theta = theta\n        self.A = (math.sin(theta), math.cos(theta))      # unit vector of +x in (E,N)\n        self.L = (-math.cos(theta), math.sin(theta))     # unit vector of +y in (E,N)\n    def from_EN(self, E, N):\n        dE, dN = np.asarray(E) - self.E0, np.asarray(N) - self.N0\n        return dE * self.A[0] + dN * self.A[1], dE * self.L[0] + dN * self.L[1]\n    def from_lonlat(self, lon, lat):\n        E, N = self.fwd.transform(np.asarray(lon), np.asarray(lat))\n        return self.from_EN(E, N)\n    def to_lonlat(self, x, y):\n        x, y = np.asarray(x), np.asarray(y)\n        E = self.E0 + x * self.A[0] + y * self.L[0]\n        N = self.N0 + x * self.A[1] + y * self.L[1]\n        return self.inv.transform(E, N)\n\nframe = Frame((minLon + maxLon) / 2, (minLat + maxLat) / 2)\nif args.align_st_charles:\n    best, theta = 0, None\n    for f in feats:\n        p = props_of(f); g = f['geometry']\n        if g['type'] == 'LineString' and (p['tags'].get('highway') or p['tags'].get('railway')) and 'charles' in (p['tags'].get('name') or '').lower():\n            c = g['coordinates']; E0, N0 = frame.fwd.transform(c[0][0], c[0][1]); E1, N1 = frame.fwd.transform(c[-1][0], c[-1][1])\n            dE, dN = E1 - E0, N1 - N0; L = math.hypot(dE, dN)\n            if L > best:\n                best = L\n                if dE < 0: dE, dN = -dE, -dN\n                theta = math.atan2(dE, dN)\n    if theta is not None:\n        frame.set_theta(theta); log(f'aligned frame to St. Charles Avenue: bearing {math.degrees(theta):.1f} deg')\n    else:\n        log('no St. Charles Avenue way found; keeping north-up frame')\n\ncorners = np.array([frame.from_lonlat(lo, la) for lo, la in [(minLon, minLat), (maxLon, minLat), (maxLon, maxLat), (minLon, maxLat)]])\nX0, X1 = corners[:, 0].min() - 20, corners[:, 0].max() + 20\nY0, Y1 = corners[:, 1].min() - 20, corners[:, 1].max() + 20\nlog(f'local extent x {X0:.0f}..{X1:.0f} m, y {Y0:.0f}..{Y1:.0f} m')\n\ndef to_export(V):\n    \"\"\"local (x east, y north, z up) -> glTF (x, y up, z south)\"\"\"\n    return np.column_stack([V[:, 0], V[:, 2], -V[:, 1]])\n\n# ----------------------------------------------------------------------------- orthophoto\nnaip = None\nif os.path.exists(args.naip) and os.path.exists(os.path.splitext(args.naip)[0] + '.json'):\n    from PIL import Image\n    Image.MAX_IMAGE_PIXELS = None\n    img = Image.open(args.naip).convert('RGB')\n    meta = json.load(open(os.path.splitext(args.naip)[0] + '.json'))\n    naip = dict(arr=np.asarray(img), bbox=meta['bbox'], img=img)\n    log(f'orthophoto {img.size[0]}x{img.size[1]} px')\n\ndef sample_naip(x, y):\n    if naip is None:\n        return None\n    lon, lat = frame.to_lonlat(x, y)\n    b = naip['bbox']; H, W = naip['arr'].shape[:2]\n    px = np.clip(((lon - b[0]) / (b[2] - b[0]) * W).astype(int), 0, W - 1)\n    py = np.clip(((b[3] - lat) / (b[3] - b[1]) * H).astype(int), 0, H - 1)\n    return naip['arr'][py, px]\n\ndef uv_naip(x, y):\n    lon, lat = frame.to_lonlat(x, y); b = naip['bbox']\n    return np.column_stack([(lon - b[0]) / (b[2] - b[0]), (lat - b[1]) / (b[3] - b[1])])\n\n# ----------------------------------------------------------------------------- lidar\ndef read_lidar(path):\n    import laspy\n    files = sorted(glob.glob(os.path.join(path, '*.la[sz]'))) if os.path.isdir(path) else ([path] if os.path.exists(path) else [])\n    if not files:\n        return None\n    xs, ys, zs, cs = [], [], [], []\n    for fp in files:\n        with laspy.open(fp) as f:\n            crs = None\n            try: crs = f.header.parse_crs()\n            except Exception: pass\n            if crs is None and args.lidar_epsg: crs = CRS.from_epsg(args.lidar_epsg)\n            if crs is None:\n                log(f'!! {os.path.basename(fp)}: no CRS in header; pass --lidar-epsg (e.g. 26915 for UTM15N m, 3452 for Louisiana South ftUS)'); continue\n            hcrs = crs.sub_crs_list[0] if crs.is_compound else crs\n            unit = (hcrs.axis_info[0].unit_name or '').lower()\n            zscale = args.z_scale if args.z_scale else (0.3048006096 if 'foot' in unit or 'feet' in unit else 1.0)\n            tr = Transformer.from_crs(hcrs, frame.utm, always_xy=True)\n            n_keep = 0\n            for chunk in f.chunk_iterator(3_000_000):\n                E, N = tr.transform(np.asarray(chunk.x), np.asarray(chunk.y))\n                x, y = frame.from_EN(E, N)\n                keep = (x >= X0) & (x <= X1) & (y >= Y0) & (y <= Y1)\n                if not keep.any(): continue\n                xs.append(x[keep].astype(np.float32)); ys.append(y[keep].astype(np.float32))\n                zs.append((np.asarray(chunk.z)[keep] * zscale).astype(np.float32))\n                cs.append(np.asarray(chunk.classification)[keep].astype(np.uint8)); n_keep += int(keep.sum())\n            log(f'{os.path.basename(fp)}: {f.header.point_count:,} pts, {n_keep:,} inside bbox, CRS {hcrs.name}, z x{zscale:g}')\n    if not xs:\n        return None\n    x, y, z, c = np.concatenate(xs), np.concatenate(ys), np.concatenate(zs), np.concatenate(cs)\n    if len(x) > args.max_points:\n        sel = np.random.default_rng(1).choice(len(x), args.max_points, replace=False)\n        x, y, z, c = x[sel], y[sel], z[sel], c[sel]\n    log(f'point cloud: {len(x):,} points; classes present: {sorted(set(np.unique(c).tolist()))}')\n    return x, y, z, c\n\nclass Grid:\n    def __init__(self, res):\n        self.res = res; self.x0, self.y0 = X0, Y0\n        self.nx = int(math.ceil((X1 - X0) / res)); self.ny = int(math.ceil((Y1 - Y0) / res))\n    def idx(self, x, y):\n        ix = np.floor((x - self.x0) / self.res).astype(np.int64); iy = np.floor((y - self.y0) / self.res).astype(np.int64)\n        ok = (ix >= 0) & (ix < self.nx) & (iy >= 0) & (iy < self.ny)\n        return ix, iy, ok\n    def max(self, x, y, z):\n        ix, iy, ok = self.idx(x, y); out = np.full(self.nx * self.ny, -np.inf, np.float32)\n        np.maximum.at(out, iy[ok] * self.nx + ix[ok], z[ok]); out[np.isinf(out)] = np.nan\n        return out.reshape(self.ny, self.nx)\n    def min(self, x, y, z):\n        ix, iy, ok = self.idx(x, y); out = np.full(self.nx * self.ny, np.inf, np.float32)\n        np.minimum.at(out, iy[ok] * self.nx + ix[ok], z[ok]); out[np.isinf(out)] = np.nan\n        return out.reshape(self.ny, self.nx)\n    def mean(self, x, y, z):\n        ix, iy, ok = self.idx(x, y); s = np.zeros(self.nx * self.ny); n = np.zeros(self.nx * self.ny)\n        np.add.at(s, iy[ok] * self.nx + ix[ok], z[ok].astype(np.float64)); np.add.at(n, iy[ok] * self.nx + ix[ok], 1)\n        with np.errstate(invalid='ignore', divide='ignore'): m = s / n\n        m[n == 0] = np.nan; return m.reshape(self.ny, self.nx)\n    def count(self, x, y):\n        ix, iy, ok = self.idx(x, y); n = np.zeros(self.nx * self.ny, np.int32); np.add.at(n, iy[ok] * self.nx + ix[ok], 1); return n.reshape(self.ny, self.nx)\n    def centers(self, i0=0, i1=None, j0=0, j1=None):\n        i1 = self.ny if i1 is None else i1; j1 = self.nx if j1 is None else j1\n        cy, cx = np.mgrid[i0:i1, j0:j1]\n        return self.x0 + (cx + 0.5) * self.res, self.y0 + (cy + 0.5) * self.res\n\ndef fill_nearest(a):\n    m = np.isnan(a)\n    if m.all(): return np.zeros_like(a)\n    if not m.any(): return a\n    idx = ndimage.distance_transform_edt(m, return_distances=False, return_indices=True)\n    return a[tuple(idx)]\n\ncloud = read_lidar(args.lidar)\nG = Grid(args.res); GT = Grid(args.terrain_res)\nif cloud is not None:\n    x, y, z, c = cloud\n    classified = bool(((c == 2) | (c == 6) | (c == 5)).any())\n    good = ~np.isin(c, [7, 18])                                  # drop noise classes\n    DSM = G.max(x[good], y[good], z[good])\n    if classified and (c == 2).any():\n        DTM = fill_nearest(GT.mean(x[c == 2], y[c == 2], z[c == 2])); DTM = ndimage.median_filter(DTM, 5)\n    else:\n        DTM = fill_nearest(GT.min(x[good], y[good], z[good])); DTM = ndimage.uniform_filter(ndimage.minimum_filter(DTM, 7), 7)\n    DSM6 = G.max(x[c == 6], y[c == 6], z[c == 6]) if classified and (c == 6).any() else None\n    VEG = G.count(x[np.isin(c, [3, 4, 5])], y[np.isin(c, [3, 4, 5])]) if classified else None\n    BLD = G.count(x[c == 6], y[c == 6]) if classified else None\n    log(f'DSM {G.nx}x{G.ny} @ {G.res} m, DTM {GT.nx}x{GT.ny} @ {GT.res} m, classified={classified}')\n    zoom = (G.ny / GT.ny, G.nx / GT.nx)\n    DTM_fine = ndimage.zoom(DTM, zoom, order=1)[:G.ny, :G.nx]\n    if DTM_fine.shape != DSM.shape:\n        DTM_fine = np.pad(DTM_fine, ((0, G.ny - DTM_fine.shape[0]), (0, G.nx - DTM_fine.shape[1])), mode='edge')\nelse:\n    log('no LiDAR found \u2014 building LOD1 blocks from tags/estimates only (run fetch_lidar.py for the real scan)')\n    DSM = DTM = DSM6 = VEG = BLD = None; classified = False\n    DTM = np.zeros((GT.ny, GT.nx), np.float32); DTM_fine = np.zeros((G.ny, G.nx), np.float32)\n\n# ----------------------------------------------------------------------------- mesh helpers\ndef heightfield_mesh(mask, top, base_z, x0, y0, res, walls=True):\n    \"\"\"Continuous surface over the cells in `mask` (corner heights = mean of adjacent cells), plus optional\n    vertical walls down to base_z around the region. Returns (V local xyz, F, is_top per vertex).\"\"\"\n    ny, nx = mask.shape\n    tops = np.where(mask, top, np.nan).astype(np.float64)\n    s = np.zeros((ny + 1, nx + 1)); n = np.zeros((ny + 1, nx + 1))\n    val = np.nan_to_num(tops); cnt = (~np.isnan(tops)).astype(np.float64)\n    for dy in (0, 1):\n        for dx in (0, 1):\n            s[dy:dy + ny, dx:dx + nx] += val; n[dy:dy + ny, dx:dx + nx] += cnt\n    with np.errstate(invalid='ignore', divide='ignore'): H = s / n\n    H = np.nan_to_num(H)\n    cy, cx = np.mgrid[0:ny + 1, 0:nx + 1]\n    V = np.column_stack([(x0 + cx * res).ravel(), (y0 + cy * res).ravel(), H.ravel()])\n    vid = np.arange((ny + 1) * (nx + 1)).reshape(ny + 1, nx + 1)\n    ci, cj = np.nonzero(mask)\n    v00, v01, v11, v10 = vid[ci, cj], vid[ci, cj + 1], vid[ci + 1, cj + 1], vid[ci + 1, cj]\n    F = [np.stack([v00, v01, v11], 1), np.stack([v00, v11, v10], 1)]\n    is_top = np.ones(len(V), bool)\n    if walls:\n        pad = np.pad(mask, 1)\n        extra_V, extra_F = [], []\n        nv = len(V)\n        # (neighbor offset, corner A, corner B) in CCW order around the cell\n        for (di, dj), A, B in [((-1, 0), (0, 0), (0, 1)), ((0, 1), (0, 1), (1, 1)), ((1, 0), (1, 1), (1, 0)), ((0, -1), (1, 0), (0, 0))]:\n            nb = pad[1 + di:1 + di + ny, 1 + dj:1 + dj + nx]\n            ei, ej = np.nonzero(mask & ~nb)\n            if len(ei) == 0: continue\n            At = vid[ei + A[0], ej + A[1]]; Bt = vid[ei + B[0], ej + B[1]]\n            Ab = np.column_stack([V[At, 0], V[At, 1], np.full(len(At), base_z)]); Bb = np.column_stack([V[Bt, 0], V[Bt, 1], np.full(len(Bt), base_z)])\n            iAb = nv + np.arange(len(At)); iBb = iAb + len(At); nv += 2 * len(At)\n            extra_V += [Ab, Bb]; extra_F += [np.stack([iAb, iBb, Bt], 1), np.stack([iAb, Bt, At], 1)]\n        if extra_V:\n            V = np.vstack([V] + extra_V); F += extra_F; is_top = np.concatenate([is_top, np.zeros(len(V) - len(is_top), bool)])\n    F = np.vstack(F)\n    used = np.unique(F); remap = np.full(len(V), -1, np.int64); remap[used] = np.arange(len(used))\n    return V[used], remap[F], is_top[used]\n\ndef to_trimesh(V, F, colors=None, name=None):\n    m = trimesh.Trimesh(vertices=to_export(V), faces=F, process=False)\n    if colors is not None: m.visual.vertex_colors = colors\n    if name: m.metadata['name'] = name\n    return m\n\nWALL = np.array([214, 206, 194, 255], np.uint8)\n\ndef color_vertices(V, is_top, fallback=(200, 120, 100)):\n    cols = np.tile(WALL, (len(V), 1))\n    if is_top.any():\n        s = sample_naip(V[is_top, 0], V[is_top, 1])\n        if s is None: s = np.tile(np.array(fallback, np.uint8), (int(is_top.sum()), 1))\n        cols[is_top, :3] = s\n    return cols\n\n# ----------------------------------------------------------------------------- buildings\nscene_b = trimesh.Scene()\nbuilding_mask_all = np.zeros((G.ny, G.nx), bool)\nenriched = []; stats = dict(buildings=0, measured=0, lod2=0, lod1=0, skipped=0)\ncx_all, cy_all = G.centers()\n\nfor k, f in enumerate(feats):\n    p = props_of(f); g = f['geometry']\n    if not p['building'] or g['type'] not in ('Polygon', 'MultiPolygon'):\n        enriched.append(f); continue\n    try:\n        geom = shape(g)\n    except Exception:\n        enriched.append(f); continue\n    if geom.is_empty: continue\n    # to local coords\n    def _tf(geom):\n        return shapely.transform(geom, lambda a: np.column_stack(frame.from_lonlat(a[:, 0], a[:, 1])))\n    loc = _tf(geom)\n    if not loc.is_valid: loc = loc.buffer(0)\n    if loc.is_empty or loc.area < 3: stats['skipped'] += 1; continue\n    bx0, by0, bx1, by1 = loc.bounds\n    j0 = max(0, int((bx0 - G.x0) / G.res) - 1); j1 = min(G.nx, int((bx1 - G.x0) / G.res) + 2)\n    i0 = max(0, int((by0 - G.y0) / G.res) - 1); i1 = min(G.ny, int((by1 - G.y0) / G.res) + 2)\n    if j1 <= j0 or i1 <= i0: stats['skipped'] += 1; continue\n    cx, cy = G.centers(i0, i1, j0, j1)\n    mask = shapely.contains_xy(loc, cx, cy)\n    name = p['name'] or f\"{p['building']}_{p['id'] or k}\"\n    base = float(np.nanmedian(DTM_fine[i0:i1, j0:j1][mask])) if (mask.any() and cloud is not None) else 0.0\n    height = None; source = 'estimate'; eave = ridge = None; top = None\n    if cloud is not None and mask.any():\n        roofgrid = DSM[i0:i1, j0:j1].copy()\n        if DSM6 is not None:\n            r6 = DSM6[i0:i1, j0:j1]\n            if np.isfinite(r6[mask]).mean() >= 0.3: roofgrid = np.where(np.isfinite(r6), r6, np.nan)\n        vals = roofgrid[mask] - base; vals = vals[np.isfinite(vals)]\n        if len(vals) >= 3:\n            ridge = float(np.percentile(vals, 97)); eave = float(np.percentile(vals, 20)); height = max(2.5, ridge); source = 'lidar'; stats['measured'] += 1\n            top = np.where(mask, roofgrid, np.nan); top = fill_nearest(top); top = ndimage.median_filter(top, 3)\n            top = np.clip(top, base + max(2.0, eave - 1.0), base + ridge + 1.5)\n    if height is None:\n        height = p['height'] if p['height'] else (p['levels'] * 3.5 + 1.0 if p['levels'] else {'house': 7.5, 'residential': 7.5, 'garage': 3.5, 'shed': 3.0, 'church': 13.0, 'university': 14.0, 'school': 11.0, 'dormitory': 16.0, 'stadium': 18.0}.get(str(p['building']), 9.0))\n        source = 'tag' if (p['height'] or p['levels']) else 'estimate'\n        top = np.full(mask.shape, base + height)\n    building_mask_all[i0:i1, j0:j1] |= mask\n    # mesh\n    mesh = None\n    if mask.sum() >= 2:\n        V, F, is_top = heightfield_mesh(mask, top, base - 0.6, G.x0 + j0 * G.res, G.y0 + i0 * G.res, G.res, walls=True)\n        mesh = to_trimesh(V, F, color_vertices(V, is_top), name); stats['lod2' if source == 'lidar' else 'lod1'] += 1\n    else:  # tiny footprint: exact extrusion of the polygon\n        try:\n            polys = [loc] if isinstance(loc, Polygon) else list(loc.geoms)\n            parts = [trimesh.creation.extrude_polygon(pg, height) for pg in polys if pg.area > 0.5]\n            if parts:\n                mesh = trimesh.util.concatenate(parts); mesh.apply_translation([0, 0, base - 0.6])\n                mesh = trimesh.Trimesh(vertices=to_export(mesh.vertices), faces=mesh.faces, process=False)\n                mesh.visual.vertex_colors = np.tile(WALL, (len(mesh.vertices), 1)); stats['lod1'] += 1\n        except Exception as e:\n            log(f'extrude failed for {name}: {e}')\n    if mesh is None: stats['skipped'] += 1; continue\n    stats['buildings'] += 1\n    scene_b.add_geometry(mesh, geom_name=f'{name}#{k}', node_name=f'{name}#{k}')\n    props = dict(f.get('properties') or {})\n    tags = props['tags'] if isinstance(props.get('tags'), dict) else props\n    tags.update({'height': round(float(height), 1), 'height:source': source, 'name': name if p['name'] else tags.get('name')})\n    if p['name'] is None: tags.pop('name', None)\n    if eave is not None: tags['roof:height'] = round(max(0.0, ridge - eave), 1); tags['est:eave'] = round(eave, 1)\n    tags['est:base'] = round(base, 2)\n    f2 = dict(f); f2['properties'] = props; enriched.append(f2)\n    if stats['buildings'] % 250 == 0: log(f\"  {stats['buildings']} buildings so far \u2026\")\nlog(f\"buildings: {stats}\")\n\n# ----------------------------------------------------------------------------- trees\ntrees = []\nif cloud is not None:\n    CHM = DSM - DTM_fine\n    if VEG is not None:\n        veg = (VEG > 0) & (VEG >= BLD) & ~building_mask_all\n    else:\n        veg = (CHM > 2.5) & ~ndimage.binary_dilation(building_mask_all, iterations=2)\n    chm = np.where(veg & np.isfinite(CHM), CHM, 0.0)\n    sm = ndimage.gaussian_filter(chm, 1.0 / G.res)\n    win = max(3, int(round(7.0 / G.res)) | 1)\n    peaks = (sm == ndimage.maximum_filter(sm, size=win)) & (sm > 3.0) & veg\n    pi, pj = np.nonzero(peaks)\n    h = chm[pi, pj]; order = np.argsort(-h)[:args.max_trees]\n    for i, j in zip(pi[order], pj[order]):\n        hh = float(chm[i, j]); trees.append(dict(x=float(G.x0 + (j + 0.5) * G.res), y=float(G.y0 + (i + 0.5) * G.res), z=float(DTM_fine[i, j]), height=round(hh, 1), radius=round(float(np.clip(0.35 * hh, 1.5, 7.0)), 1)))\n    log(f'trees detected from canopy: {len(trees)}')\n\n# ----------------------------------------------------------------------------- exports\ndef export(scene_or_mesh, name):\n    path = os.path.join(args.out, name + '.glb'); scene_or_mesh.export(path)\n    if args.obj:\n        try: scene_or_mesh.export(os.path.join(args.out, name + '.obj'))\n        except Exception as e: log(f'obj export of {name} failed: {e}')\n    log(f'wrote {path} ({os.path.getsize(path) / 1048576:.1f} MB)')\n\nif stats['buildings']:\n    export(scene_b, 'tulane_buildings')\n\n# terrain\nmask_t = np.ones(DTM.shape, bool)\nV, F, _ = heightfield_mesh(mask_t, DTM, 0.0, GT.x0, GT.y0, GT.res, walls=False)\nterrain = trimesh.Trimesh(vertices=to_export(V), faces=F, process=False)\nif naip is not None:\n    terrain.visual = trimesh.visual.TextureVisuals(uv=uv_naip(V[:, 0], V[:, 1]), image=naip['img'])\nelse:\n    terrain.visual.vertex_colors = np.tile(np.array([158, 205, 96, 255], np.uint8), (len(V), 1))\nterrain.metadata['name'] = 'terrain'\nexport(terrain, 'tulane_terrain')\n\n# trees: instanced crowns + trunks\nif trees:\n    scene_t = trimesh.Scene()\n    crowns = {}\n    for tint, col in (('light', [120, 178, 78, 255]), ('mid', [86, 152, 62, 255]), ('dark', [64, 122, 48, 255])):\n        m = trimesh.creation.icosphere(subdivisions=1, radius=1.0); m.visual.face_colors = np.tile(np.array(col, np.uint8), (len(m.faces), 1)); crowns[tint] = m\n    trunk = trimesh.creation.cylinder(radius=1.0, height=1.0, sections=8); trunk.visual.face_colors = np.tile(np.array([111, 75, 51, 255], np.uint8), (len(trunk.faces), 1))\n    for tint, m in crowns.items(): scene_t.geometry[f'crown_{tint}'] = m      # one shared mesh per tint, instanced by nodes\n    scene_t.geometry['trunk'] = trunk\n    Rx = trimesh.transformations.rotation_matrix(math.pi / 2, [1, 0, 0])  # cylinder axis z -> y\n    for i, t in enumerate(trees):\n        tint = 'light' if t['height'] < 8 else ('mid' if t['height'] < 14 else 'dark')\n        r = t['radius']; ch = max(2.0, t['height'] * 0.6); cz = t['z'] + t['height'] - ch / 2\n        Tc = np.diag([r, ch / 2, r, 1.0]); Tc[:3, 3] = [t['x'], cz, -t['y']]\n        scene_t.graph.update(frame_to=f'tree_{i}', frame_from='world', matrix=Tc, geometry=f'crown_{tint}')\n        th = max(0.5, t['height'] - ch); Tt = np.eye(4); Tt[:3, :3] = np.diag([r * 0.12, th, r * 0.12]); Tt[:3, 3] = [t['x'], t['z'] + th / 2, -t['y']]\n        scene_t.graph.update(frame_to=f'trunk_{i}', frame_from='world', matrix=Tt @ Rx, geometry='trunk')\n    export(scene_t, 'tulane_trees')\n    with open(os.path.join(args.out, 'tulane_trees.json'), 'w') as fh:\n        json.dump(dict(frame='meters; x east (or along St. Charles with --align), y north, z = ground elevation; glTF uses (x, up, -y)', trees=trees), fh)\n\n# whole scanned surface\nif args.surface and cloud is not None:\n    GS = Grid(args.surface_res); good = ~np.isin(c, [7, 18]); S = fill_nearest(GS.max(x[good], y[good], z[good]))\n    V, F, _ = heightfield_mesh(np.ones(S.shape, bool), S, 0.0, GS.x0, GS.y0, GS.res, walls=False)\n    surf = trimesh.Trimesh(vertices=to_export(V), faces=F, process=False)\n    if naip is not None: surf.visual = trimesh.visual.TextureVisuals(uv=uv_naip(V[:, 0], V[:, 1]), image=naip['img'])\n    else: surf.visual.vertex_colors = np.tile(np.array([190, 190, 180, 255], np.uint8), (len(V), 1))\n    export(surf, 'tulane_scan_surface')\n\n# enriched geojson (loads into the voxel viewer; heights now measured)\nwith open(os.path.join(args.out, 'tulane_enriched.geojson'), 'w', encoding='utf-8') as fh:\n    json.dump(dict(type='FeatureCollection', features=enriched), fh)\nlog('wrote tulane_enriched.geojson')\n\n# preview\ntry:\n    import matplotlib; matplotlib.use('Agg'); import matplotlib.pyplot as plt; from matplotlib.colors import LightSource\n    fig, ax = plt.subplots(figsize=(10, 10 * (Y1 - Y0) / (X1 - X0)))\n    if DSM is not None:\n        ls = LightSource(azdeg=315, altdeg=45); ax.imshow(ls.shade(fill_nearest(DSM), cmap=plt.cm.gray, vert_exag=1.5, blend_mode='soft'), extent=[X0, X1, Y0, Y1], origin='lower')\n    for f in enriched:\n        g = f['geometry']; pp = props_of(f)\n        if pp['building'] and g['type'] in ('Polygon', 'MultiPolygon'):\n            for poly in ([g['coordinates']] if g['type'] == 'Polygon' else g['coordinates']):\n                ring = np.array(poly[0]); lx, ly = frame.from_lonlat(ring[:, 0], ring[:, 1]); ax.plot(lx, ly, color='#ff6a3d', lw=0.5)\n    ax.set_aspect('equal'); ax.set_title('LiDAR surface (hillshade) with footprints'); fig.savefig(os.path.join(args.out, 'preview_dsm.png'), dpi=130, bbox_inches='tight')\n    log('wrote preview_dsm.png')\nexcept Exception as e:\n    log(f'preview skipped: {e}')\n\nstats.update(trees=len(trees), lidar=cloud is not None, orthophoto=naip is not None, frame='aligned to St. Charles' if args.align_st_charles else 'north-up')\njson.dump(stats, open(os.path.join(args.out, 'model_stats.json'), 'w'), indent=1)\nlog('done', stats)\n"
}
for name, text in FILES.items():
    open(name, 'w', encoding='utf-8').write(text)
print('pipeline files written:', ', '.join(FILES))

## 2. Install and fetch footprints + aerial photo

In [ ]:
%cd /content/tulane-scan
%pip -q install -r requirements.txt
!python fetch_osm.py
!python fetch_naip.py

## 3. Which LiDAR scans cover the campus?
Read the list: the newest project is chosen automatically. If you want a different one, put part of its name in `PROJECT` in step 4.

In [ ]:
!python fetch_lidar.py --list

## 4. Download the scan and build the model

In [ ]:
PROJECT = ''   # optional: text from a project name in step 3, e.g. 'LA_Southeast'
ROOF_RES = 0.5 # meters; 1.0 is faster, 0.5 gives cleaner building edges
!python fetch_lidar.py --yes {('--project ' + PROJECT) if PROJECT else ''}
!python build_model.py --align-st-charles --surface --res {ROOF_RES} --obj
from IPython.display import Image, display
import json, os
print(json.load(open('out/model_stats.json')))
display(Image('out/preview_dsm.png', width=700))

## 5. Save to Google Drive
Creates `TulaneCampus3D` in your Drive with the .glb/.obj models, the enriched GeoJSON (for the voxel viewer) and the preview.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/TulaneCampus3D && cp -r out/* /content/drive/MyDrive/TulaneCampus3D/ && ls -la /content/drive/MyDrive/TulaneCampus3D
print('Saved. Open the .glb files in Blender/Unity/Godot, or load tulane_enriched.geojson into tulane_voxel_campus.html')